# Recomendação de Planos Megaline: Smart ou Ultra

## Introdução

A operadora de celular **Megaline** identificou que muitos de seus clientes ainda utilizam planos antigos. Para incentivar a migração, a empresa deseja desenvolver um modelo de **machine learning** capaz de analisar o comportamento mensal dos clientes e recomendar automaticamente um dos dois planos mais recentes: **Smart** ou **Ultra**.

## Objetivo

Construir um modelo de **classificação binária** que, a partir dos dados de comportamento dos clientes (chamadas, minutos, mensagens e tráfego de internet), preveja qual plano é mais adequado para cada usuário.

- **Variável alvo (`is_ultra`):**
  - `0` → Plano Smart
  - `1` → Plano Ultra

## Critério de sucesso

O modelo deve atingir uma **acurácia mínima de 0,75** no conjunto de teste.

## Etapas do projeto

1. Carregar e examinar os dados.
2. Dividir os dados em conjuntos de treinamento, validação e teste.
3. Treinar diferentes modelos de classificação, ajustando hiperparâmetros.
4. Avaliar o desempenho dos modelos no conjunto de validação.
5. Selecionar o melhor modelo e testá-lo no conjunto de teste.
6. (Tarefa adicional) Realizar uma prova de sanidade do modelo.

## Carregamento e análise de dados

In [1]:
# Carregando bibliotecas
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
# Carregando o dataframe
df = pd.read_csv('/datasets/users_behavior.csv')
print(df.sample(10))
print('_________________________________________________')
print(df.info())

      calls  minutes  messages   mb_used  is_ultra
,576    72.0   498.89      71.0  17450.90         1
,1344   69.0   536.63      47.0  15988.41         0
,2168   66.0   443.44      43.0  21871.05         0
,2652   75.0   539.43      83.0  14021.90         1
,1619   96.0   620.43      37.0  12569.40         0
,2166   27.0   182.90      15.0  12756.92         0
,543    66.0   432.02      54.0  24008.36         0
,1633   87.0   638.31      76.0  28239.49         1
,1046   70.0   533.32      13.0  10801.91         0
,2374   43.0   296.15      59.0  15266.71         0
,_________________________________________________
,<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 3214 entries, 0 to 3213
,Data columns (total 5 columns):
, #   Column    Non-Null Count  Dtype  
,---  ------    --------------  -----  
, 0   calls     3214 non-null   float64
, 1   minutes   3214 non-null   float64
, 2   messages  3214 non-null   float64
, 3   mb_used   3214 non-null   float64
, 4   is_ultra  3214 non-null

In [3]:
# Estatísticas descritivas separadas por plano
print('MÉDIA DOS SERVIÇOS DE CADA PLANO (Smart = 0 , Ultra = 1)')
print('')
print(df.groupby('is_ultra').mean())
print('_____________________________________________________________________')
print('DESCRIÇÃO COMPLETA DOS SERVIÇOS DE CADA PLANO (Smart = 0 , Ultra = 1)')
print('')
print(df.groupby('is_ultra').describe().T)

MÉDIA DOS SERVIÇOS DE CADA PLANO (Smart = 0 , Ultra = 1)
,
,              calls     minutes   messages       mb_used
,is_ultra                                                
,0         58.463437  405.942952  33.384029  16208.466949
,1         73.392893  511.224569  49.363452  19468.823228
,_____________________________________________________________________
,DESCRIÇÃO COMPLETA DOS SERVIÇOS DE CADA PLANO (Smart = 0 , Ultra = 1)
,
,is_ultra                   0             1
,calls    count   2229.000000    985.000000
,         mean      58.463437     73.392893
,         std       25.939858     43.916853
,         min        0.000000      0.000000
,         25%       40.000000     41.000000
,         50%       60.000000     74.000000
,         75%       76.000000    104.000000
,         max      198.000000    244.000000
,minutes  count   2229.000000    985.000000
,         mean     405.942952    511.224569
,         std      184.512604    308.031100
,         min        0.000000      0.

In [4]:
# Verificando quantos clientes em cada plano
print('Quantidade de clientes em cada plano (Smart = 0 , Ultra = 1)')
print(df['is_ultra'].value_counts())
print('_____________________________________________________________')
print('Proporção de clientes em cada plano (Smart = 0 , Ultra = 1)')
print(df['is_ultra'].value_counts(normalize=True))

Quantidade de clientes em cada plano (Smart = 0 , Ultra = 1)
,0    2229
,1     985
,Name: is_ultra, dtype: int64
,_____________________________________________________________
,Proporção de clientes em cada plano (Smart = 0 , Ultra = 1)
,0    0.693528
,1    0.306472
,Name: is_ultra, dtype: float64


## Conclusão da análise inicial

A análise exploratória dos dados revelou os seguintes pontos:

- O dataset contém **3214 registros** e **5 colunas**, sem valores nulos, confirmando que o pré-processamento já foi realizado corretamente.
- A variável alvo (`is_ultra`) é **moderadamente desbalanceada**: aproximadamente **69,4%** dos clientes utilizam o plano **Smart** e **30,6%** utilizam o plano **Ultra**. Esse desbalanceamento será levado em conta na avaliação dos modelos.
- Ao comparar as estatísticas de uso entre os dois planos, observa-se que os clientes do plano **Ultra** apresentam, em média, valores mais altos em todas as features:
  - Mais chamadas (73,4 vs 58,5 em média)
  - Mais minutos de uso (511,2 vs 405,9 em média)
  - Mais mensagens enviadas (49,4 vs 33,4 em média)
  - Maior consumo de internet (19.468,8 MB vs 16.208,5 MB em média)
- Essas diferenças indicam que existe um padrão de comportamento real associado a cada plano, o que é um bom indício de que um modelo de classificação conseguirá aprender a distinguir os grupos com base nessas features.

Com os dados validados e compreendidos, podemos seguir para a divisão dos conjuntos de treinamento, validação e teste, e em seguida para a construção e comparação dos modelos.

## Dividindo os dados em conjuntos de treinamento, validação e teste.

In [5]:
# Dividindo os dados em conjuntos de treinamento, validação e teste.

# Separando features (X) e target (y)
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

# Primeira divisão: 60% treino, 40% restante
features_train, features_rest, target_train, target_rest = train_test_split(
    features, target, test_size=0.4, random_state=12345)

# Segunda divisão: dividindo o restante em validação (20%) e teste (20%)
features_valid, features_test, target_valid, target_test = train_test_split(
    features_rest, target_rest, test_size=0.5, random_state=12345)

# Verificando os tamanhos
print('Treino:', features_train.shape, target_train.shape)
print('Validação:', features_valid.shape, target_valid.shape)
print('Teste:', features_test.shape, target_test.shape)

Treino: (1928, 4) (1928,)
,Validação: (643, 4) (643,)
,Teste: (643, 4) (643,)


In [6]:
# MODELO 1 - ÁRVORE DE DECISÃO

print('Resultados - Árvore de Decisão')
print('depth : accuracy')

best_tree_score = 0
best_tree_depth = 0

for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    score = accuracy_score(target_valid, predictions_valid)
    
    print(f'{depth} : {score:.4f}')
    
    if score > best_tree_score:
        best_tree_score = score
        best_tree_depth = depth

print()
print(f'Best depth: {best_tree_depth}, Accuracy: {best_tree_score:.4f}')

Resultados - Árvore de Decisão
,depth : accuracy
,1 : 0.7543
,2 : 0.7823
,3 : 0.7854
,4 : 0.7792
,5 : 0.7792
,6 : 0.7838
,7 : 0.7823
,8 : 0.7792
,9 : 0.7823
,10 : 0.7745
,
,Best depth: 3, Accuracy: 0.7854


### Árvore de Decisão

Foram testadas profundidades (`max_depth`) variando de 1 a 10, avaliando a acurácia no conjunto de validação:

| max_depth | accuracy |
|-----------|----------|
| 1         | 0,7543   |
| 2         | 0,7823   |
| 3         | **0,7854** |
| 4         | 0,7792   |
| 5         | 0,7792   |
| 6         | 0,7838   |
| 7         | 0,7823   |
| 8         | 0,7792   |
| 9         | 0,7823   |
| 10        | 0,7745   |

**Melhor resultado:** `max_depth = 3`, com acurácia de **0,7854** no conjunto de validação.

**Observações:**
- Já com profundidade 1 o modelo supera o limite mínimo de 0,75 exigido pelo projeto.
- A acurácia melhora rapidamente até `max_depth = 3`, ponto em que a árvore atinge o melhor equilíbrio entre complexidade e capacidade de generalização.
- A partir de `max_depth = 4`, a acurácia oscila e não apresenta ganhos consistentes, indicando início de **overfitting** — a árvore passa a se ajustar demais aos dados de treino, perdendo um pouco de capacidade de generalização no conjunto de validação.

In [7]:

# MODELO 2 - FLORESTA ALEATÓRIA

print('Resultados - Random Forest')
print('n_estimators : max_depth : accuracy')

best_forest_score = 0
best_forest_n_est = 0
best_forest_depth = 0

for n_est in range(10, 51, 10):
    for depth in range(1, 11):
        model = RandomForestClassifier(random_state=12345, n_estimators=n_est, max_depth=depth)
        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)
        score = accuracy_score(target_valid, predictions_valid)
        
        print(f'{n_est} : {depth} : {score:.4f}')
        
        if score > best_forest_score:
            best_forest_score = score
            best_forest_n_est = n_est
            best_forest_depth = depth

print()
print(f'Melhores parâmetros: n_estimators={best_forest_n_est}, max_depth={best_forest_depth}, acurácia: {best_forest_score:.4f}')


Resultados - Random Forest
,n_estimators : max_depth : accuracy
,10 : 1 : 0.7558
,10 : 2 : 0.7776
,10 : 3 : 0.7854
,10 : 4 : 0.7900
,10 : 5 : 0.7932
,10 : 6 : 0.8009
,10 : 7 : 0.7947
,10 : 8 : 0.7963
,10 : 9 : 0.7854
,10 : 10 : 0.7916
,20 : 1 : 0.7667
,20 : 2 : 0.7838
,20 : 3 : 0.7869
,20 : 4 : 0.7885
,20 : 5 : 0.7900
,20 : 6 : 0.7994
,20 : 7 : 0.8009
,20 : 8 : 0.7978
,20 : 9 : 0.7900
,20 : 10 : 0.7916
,30 : 1 : 0.7667
,30 : 2 : 0.7838
,30 : 3 : 0.7869
,30 : 4 : 0.7869
,30 : 5 : 0.7932
,30 : 6 : 0.8009
,30 : 7 : 0.8025
,30 : 8 : 0.7994
,30 : 9 : 0.7932
,30 : 10 : 0.7947
,40 : 1 : 0.7760
,40 : 2 : 0.7854
,40 : 3 : 0.7869
,40 : 4 : 0.7900
,40 : 5 : 0.7947
,40 : 6 : 0.8025
,40 : 7 : 0.8025
,40 : 8 : 0.8087
,40 : 9 : 0.7947
,40 : 10 : 0.7963
,50 : 1 : 0.7589
,50 : 2 : 0.7838
,50 : 3 : 0.7869
,50 : 4 : 0.7869
,50 : 5 : 0.7932
,50 : 6 : 0.7994
,50 : 7 : 0.8025
,50 : 8 : 0.8072
,50 : 9 : 0.7978
,50 : 10 : 0.7932
,
,Melhores parâmetros: n_estimators=40, max_depth=8, acurácia: 0.8087


### Random Forest

Foram testadas combinações de `n_estimators` (10 a 50, em passos de 10) e `max_depth` (1 a 10), avaliando a acurácia no conjunto de validação.

**Melhor resultado:** `n_estimators = 40`, `max_depth = 8`, com acurácia de **0,8087** no conjunto de validação.

**Observações:**
- Todas as combinações testadas superaram o limite mínimo de 0,75 exigido pelo projeto, demonstrando que o Random Forest é um modelo robusto para este problema.
- A acurácia tende a melhorar conforme a profundidade aumenta até a faixa de 6 a 8, estabilizando ou até caindo levemente em profundidades maiores (ex: `max_depth = 10`), o que sugere início de overfitting.
- O número de árvores (`n_estimators`) tem impacto mais discreto isoladamente, mas combinado com profundidades adequadas (6 a 8) consistentemente produz os melhores resultados.
- O Random Forest superou a Árvore de Decisão (0,7854 → 0,8087), confirmando o ganho esperado ao combinar múltiplas árvores para reduzir variância e overfitting.

In [8]:
# MODELO 3 - REGRESSÃO LOGISTICA

model = LogisticRegression(random_state=12345, solver='liblinear')
model.fit(features_train, target_train)
predictions_valid = model.predict(features_valid)
score = accuracy_score(target_valid, predictions_valid)

print(f'Acurácia - Regressão Logística: {score:.4f}')

Acurácia - Regressão Logística: 0.7092


### Regressão Logística

Foi treinado um modelo de Regressão Logística com `solver='liblinear'`, avaliado no conjunto de validação.

**Resultado:** acurácia de **0,7092**.

**Observações:**
- Este foi o único modelo que **não atingiu** o limite mínimo de 0,75 exigido pelo projeto.
- O desempenho inferior sugere que a relação entre as features (calls, minutes, messages, mb_used) e a escolha do plano não é predominantemente linear, já que a Regressão Logística assume fronteiras de decisão lineares entre as classes.
- Uma possível explicação de negócio: planos de celular costumam ter **limites fixos** de uso (minutos, mensagens, internet). É provável que o comportamento real dos clientes siga um **efeito de limiar (threshold)**: a probabilidade de trocar para o Ultra permanece baixa enquanto o consumo está dentro do limite do plano Smart, mas **salta abruptamente** quando o cliente passa a estourar esse limite com frequência. Esse tipo de comportamento em "degrau" não é bem capturado pela Regressão Logística, que assume uma transição suave e proporcional entre as variáveis e a probabilidade da classe.
- Modelos baseados em árvores (Árvore de Decisão e Random Forest) capturam esse tipo de padrão naturalmente, através de perguntas diretas como `mb_used > limite?`, o que explica seu desempenho superior.

### Comparação geral dos modelos (conjunto de validação)

| Modelo                  | Melhores hiperparâmetros            | Acurácia |
|--------------------------|--------------------------------------|----------|
| Árvore de Decisão        | `max_depth=3`                        | 0,7854   |
| Random Forest             | `n_estimators=40`, `max_depth=8`     | **0,8087** |
| Regressão Logística       | `solver='liblinear'`                 | 0,7092   |

**Conclusão da investigação:** o modelo com melhor desempenho no conjunto de validação foi o **Random Forest**, com `n_estimators=40` e `max_depth=8`, atingindo acurácia de **0,8087**. Este será o modelo escolhido para a avaliação final no conjunto de teste.


In [9]:
# Juntando treino + validação para o treinamento final
features_train_full = pd.concat([features_train, features_valid])
target_train_full = pd.concat([target_train, target_valid])

print('Treino + validação:', features_train_full.shape, target_train_full.shape)

# Treinando o modelo final com os melhores hiperparâmetros encontrados
final_model = RandomForestClassifier(random_state=12345, n_estimators=40, max_depth=8)
final_model.fit(features_train_full, target_train_full)

# Avaliando no conjunto de teste
test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

print(f'Acurácia no conjunto de teste: {test_accuracy:.4f}')

Treino + validação: (2571, 4) (2571,)
,Acurácia no conjunto de teste: 0.7994


## Avaliação final no conjunto de teste

Com base na investigação de modelos e hiperparâmetros, o **Random Forest** com `n_estimators=40` e `max_depth=8` foi selecionado como o modelo final, por apresentar a melhor acurácia no conjunto de validação (0,8087).

Para o treinamento final, os conjuntos de **treinamento e validação** foram combinados (2571 amostras), já que a validação não é mais necessária após a escolha dos hiperparâmetros. O conjunto de **teste** (643 amostras) foi mantido intocado durante todo o processo, sendo utilizado exclusivamente para esta avaliação final.

**Resultado:** acurácia de **0,7994** no conjunto de teste.

**Conclusão:**
- O modelo superou com folga o limite mínimo de 0,75 exigido pelo projeto.
- A leve queda em relação à acurácia de validação (0,8087 → 0,7994) é esperada e normal, indicando que o modelo generaliza bem para dados nunca vistos, sem sinais de overfitting significativo.

In [10]:
# Testando modelo Dummy para verificar a qualidade do modelo real

from sklearn.dummy import DummyClassifier

# Treinando o modelo baseline (sempre chuta a classe mais frequente)
dummy_model = DummyClassifier(strategy='most_frequent', random_state=12345)
dummy_model.fit(features_train_full, target_train_full)

# Avaliando no conjunto de teste
dummy_predictions = dummy_model.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f'Acurácia do modelo baseline (DummyClassifier): {dummy_accuracy:.4f}')
print(f'Acurácia do modelo final (Random Forest):      {test_accuracy:.4f}')
print(f'Ganho sobre o baseline:                        {test_accuracy - dummy_accuracy:.4f}')

Acurácia do modelo baseline (DummyClassifier): 0.6843
,Acurácia do modelo final (Random Forest):      0.7994
,Ganho sobre o baseline:                        0.1151


## Prova real do modelo (Sanity Check)

Para confirmar que o modelo final está genuinamente aprendendo padrões nos dados — e não apenas se beneficiando do desbalanceamento entre as classes —, foi utilizado um **DummyClassifier** como baseline, com a estratégia `most_frequent`: esse modelo não aprende nada, apenas chuta sempre a classe majoritária (Smart, `0`) para qualquer cliente.

| Modelo | Acurácia |
|---|---|
| Baseline (DummyClassifier) | 0,6843 |
| Random Forest (modelo final) | 0,7994 |
| **Ganho sobre o baseline** | **+0,1151** |

**Conclusão:**
- O baseline, ao sempre prever Smart, acerta apenas 68,43% dos casos — reflexo direto da proporção de clientes Smart no dataset (~69%).
- O Random Forest supera o baseline em **+11,51 pontos percentuais**, demonstrando que o modelo aprendeu padrões reais de comportamento que diferenciam clientes Smart de clientes Ultra.
- A prova real confirma que o modelo é válido e agrega valor real à tarefa de recomendação de planos.

## Conclusão Geral

O objetivo deste projeto foi desenvolver um modelo de classificação capaz de recomendar o plano mais adequado para cada cliente da Megaline — **Smart** ou **Ultra** — com base em seu comportamento mensal de uso (chamadas, minutos, mensagens e internet), atingindo uma acurácia mínima de **0,75**.

### Resumo do processo

1. **Análise dos dados:** o dataset contém 3.214 registros sem valores nulos, confirmando que o pré-processamento já havia sido realizado. A análise exploratória revelou um desbalanceamento moderado entre as classes (~69% Smart, ~31% Ultra) e diferenças claras de comportamento entre os dois grupos, com clientes Ultra apresentando médias mais altas em todas as features.

2. **Divisão dos dados:** os dados foram divididos nas proporções **60% treino / 20% validação / 20% teste**, resultando em conjuntos de 1.928, 643 e 643 amostras respectivamente.

3. **Investigação de modelos:** foram testados três modelos com diferentes hiperparâmetros, avaliados no conjunto de validação:

| Modelo | Melhores hiperparâmetros | Acurácia (validação) |
|---|---|---|
| Regressão Logística | `solver='liblinear'` | 0,7092 |
| Árvore de Decisão | `max_depth=3` | 0,7854 |
| Random Forest | `n_estimators=40`, `max_depth=8` | **0,8087** |

4. **Avaliação final:** o **Random Forest** foi selecionado como modelo final e treinado com treino + validação combinados (2.571 amostras). Avaliado no conjunto de teste, atingiu acurácia de **0,7994** — superando o limite mínimo exigido com folga.

5. **Prova real:** comparado a um modelo baseline (DummyClassifier que sempre prevê Smart), o Random Forest superou o baseline em **+11,51 pontos percentuais** (0,7994 vs 0,6843), confirmando que o modelo aprendeu padrões reais e agrega valor genuíno à tarefa de recomendação.

### Considerações finais

A Regressão Logística foi o único modelo que não atingiu o limite mínimo de acurácia, provavelmente porque a relação entre o comportamento de uso e a escolha do plano segue um **efeito de limiar**: clientes tendem a migrar para o Ultra quando passam a estourar frequentemente os limites do plano Smart, um comportamento em "degrau" que modelos lineares não capturam bem. Árvores de decisão e florestas aleatórias lidam naturalmente com esse tipo de padrão, o que explica seu desempenho superior.

O modelo final — **Random Forest com `n_estimators=40` e `max_depth=8`** — é a recomendação para uso em produção, com acurácia de **79,94%** no conjunto de teste.